In [18]:
!pip install -q numpy scikit-learn

## 1. Config — set your input file path and traversal parameters

In [19]:
!wget -O candidate_nodes.json https://raw.githubusercontent.com/AvadhootGandhe/India-runs-AI/pipline/pipeliine/candidate_nodes.json
!wget -O sample_100.jsonl https://raw.githubusercontent.com/AvadhootGandhe/India-runs-AI/pipline/pipeliine/sample.jsonl

--2026-07-02 12:29:53--  https://raw.githubusercontent.com/AvadhootGandhe/India-runs-AI/pipline/pipeliine/candidate_nodes.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 251208 (245K) [text/plain]
Saving to: ‘candidate_nodes.json’

candidate_nodes.jso 100%[===================>] 245.32K  --.-KB/s    in 0.003s  

2026-07-02 12:29:53 (72.3 MB/s) - ‘candidate_nodes.json’ saved [251208/251208]

--2026-07-02 12:29:53--  https://raw.githubusercontent.com/AvadhootGandhe/India-runs-AI/pipline/pipeliine/sample.jsonl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaitin

In [20]:
RAW_CANDIDATES_PATH = "/content/sample_100.jsonl"

CANDIDATES_PATH = "/content/candidate_nodes.json"

GRAPH_OUTPUT_PATH = "/content/graphical_representation.json"

SOURCE_CANDIDATE = "CAND_0071487"

NUM_NODES = 10

## 2. Fake / Honeypot Candidate Filtering (from `flag_profiles.py`)

Before the similarity graph is built, every candidate in the raw pool is scored for honeypot / fake-profile signals (impossible skill durations, inverted salary ranges, overlapping or future employment, YOE mismatches, etc.). Flagged candidates are removed, and the remaining clean pool is what Graph creation, DFS traversal, and Reasoning run on.

### 2.1 Detection checks

In [21]:
from datetime import datetime

TODAY = datetime(2026, 6, 25)

def check_skill_duration_exceeds_career(candidate):
    career = candidate.get("career_history", [])
    skills = candidate.get("skills", [])
    total_career = sum(r.get("duration_months", 0) or 0 for r in career)
    worst_excess, worst_skill = 0, None
    for s in skills:
        excess = (s.get("duration_months") or 0) - total_career
        if excess > worst_excess:
            worst_excess = excess
            worst_skill = s["name"]
    if worst_excess > 12:
        return True, f"{worst_skill} claims {worst_excess}m more than total career ({total_career}m)"
    return False, ""


def check_skill_duration_inflation(candidate):
    career = candidate.get("career_history", [])
    skills = candidate.get("skills", [])
    total_career = sum(r.get("duration_months", 0) or 0 for r in career)
    total_skills = sum(s.get("duration_months", 0) or 0 for s in skills)
    if total_career == 0:
        return False, ""
    ratio = total_skills / total_career
    if ratio > 10:
        return True, f"skill_sum={total_skills}m vs career={total_career}m (ratio={ratio:.1f}x)"
    return False, ""


def check_inverted_salary(candidate):
    sig = candidate.get("redrob_signals", {}) or {}
    sal = sig.get("expected_salary_range_inr_lpa", {}) or {}
    sal_min = sal.get("min") or 0
    sal_max = sal.get("max") or 0
    if sal_max > 0 and sal_min > sal_max:
        return True, f"salary min={sal_min}L > max={sal_max}L"
    return False, ""


def check_career_before_graduation(candidate):
    edu    = candidate.get("education", [])
    career = candidate.get("career_history", [])
    latest_edu_end = max((e.get("end_year") or 0 for e in edu), default=0)
    if not latest_edu_end or not career:
        return False, ""
    for r in career:
        try:
            s = datetime.strptime(r["start_date"], "%Y-%m-%d")
            if s.year < latest_edu_end - 2:
                return True, f"{r['company']} started {r['start_date']} but edu ends {latest_edu_end}"
        except (ValueError, KeyError):
            pass
    return False, ""


def check_overlapping_roles(candidate):
    career = candidate.get("career_history", [])
    parsed = []
    for r in career:
        try:
            s = datetime.strptime(r["start_date"], "%Y-%m-%d")
            e = datetime.strptime(r["end_date"], "%Y-%m-%d") if r.get("end_date") else TODAY
            parsed.append((s, e, r.get("company", "?")))
        except (ValueError, KeyError):
            pass
    parsed.sort()
    for i in range(len(parsed) - 1):
        s1, e1, c1 = parsed[i]
        s2, e2, c2 = parsed[i + 1]
        if c1 != c2 and s2 < e1:
            overlap_days = (min(e1, e2) - s2).days
            if overlap_days > 60:
                return True, f"{c1} overlaps {c2} by {overlap_days} days"
    return False, ""


def check_future_employment(candidate):
    for r in candidate.get("career_history", []):
        try:
            if datetime.strptime(r["start_date"], "%Y-%m-%d") > TODAY:
                return True, f"{r.get('company', '?')} starts {r['start_date']}"
        except (ValueError, KeyError):
            pass
    return False, ""


def check_end_before_start(candidate):
    for r in candidate.get("career_history", []):
        try:
            s = datetime.strptime(r["start_date"], "%Y-%m-%d")
            if r.get("end_date"):
                e = datetime.strptime(r["end_date"], "%Y-%m-%d")
                if e < s:
                    return True, f"{r.get('company', '?')}: end={r['end_date']} before start={r['start_date']}"
        except (ValueError, KeyError):
            pass
    return False, ""


def check_current_company_mismatch(candidate):
    profile = candidate.get("profile") or {}
    profile_company = (profile.get("current_company") or "").strip().lower()
    career          = candidate.get("career_history", [])
    if not profile_company:
        return False, ""
    current_roles = [r.get("company", "").strip().lower() for r in career if r.get("is_current")]
    if current_roles and profile_company not in current_roles:
        return True, (
            f"profile='{profile.get('current_company')}' "
            f"but current_role(s)={[r.get('company') for r in career if r.get('is_current')]}"
        )
    return False, ""


def check_proficiency_vs_assessment(candidate):
    EXPECTED = {
        "beginner":     (0,  50),
        "intermediate": (30, 75),
        "advanced":     (55, 95),
        "expert":       (70, 100),
    }
    skills      = candidate.get("skills", [])
    sig         = candidate.get("redrob_signals", {}) or {}
    assessments = sig.get("skill_assessment_scores", {}) or {}
    mismatches  = []
    for s in skills:
        score = assessments.get(s.get("name", ""))
        if score is None:
            continue
        lo, hi = EXPECTED.get(s.get("proficiency", ""), (0, 100))
        if score < lo - 20 or score > hi + 10:
            mismatches.append(f"{s['name']}(self={s.get('proficiency')},assessed={score:.0f})")
    if len(mismatches) >= 2:
        return True, "; ".join(mismatches[:3])
    return False, ""


def check_yoe_mismatch(candidate):
    profile       = candidate.get("profile") or {}
    stated        = profile.get("years_of_experience") or 0
    career        = candidate.get("career_history", [])
    career_months = sum(r.get("duration_months", 0) or 0 for r in career)
    diff          = abs((career_months / 12) - stated)
    if diff > 3:
        return True, f"stated={stated}yr vs career_sum={career_months/12:.1f}yr (diff={diff:.1f})"
    return False, ""


### 2.2 Scoring & thresholds

In [22]:
CHECKS = [
    (check_skill_duration_exceeds_career, 0.20, "skill_exceeds_career"),
    (check_skill_duration_inflation,      0.15, "skill_inflation"),
    (check_inverted_salary,               0.10, "inverted_salary"),
    (check_career_before_graduation,      0.15, "career_before_grad"),
    (check_overlapping_roles,             0.20, "overlapping_roles"),
    (check_future_employment,             0.25, "future_employment"),
    (check_end_before_start,             0.25, "end_before_start"),
    (check_current_company_mismatch,     0.20, "company_mismatch"),
    (check_proficiency_vs_assessment,    0.15, "proficiency_mismatch"),
    (check_yoe_mismatch,                 0.15, "yoe_mismatch"),
]

MAX_WEIGHT = sum(w for _, w, _ in CHECKS)

# Thresholds — default catches all confirmed anomalies (610 in 100K pool)
# Strict mode targets the challenge's stated ~80 honeypot count
THRESHOLDS = {
    "default": {"honeypot_score": 0.30, "honeypot_flags": 3,
                 "suspicious_score": 0.12, "suspicious_flags": 1},
    "strict":  {"honeypot_score": 0.55, "honeypot_flags": 5,
                "suspicious_score": 0.20, "suspicious_flags": 2},
}


def score_candidate(candidate, mode="default"):
    result     = {"candidate_id": candidate["candidate_id"]}
    total      = 0.0
    all_details = []

    for fn, weight, label in CHECKS:
        flagged, detail = fn(candidate)
        result[label]            = flagged
        result[f"{label}_detail"] = detail
        if flagged:
            total += weight
            all_details.append(f"[{label}] {detail}")

    result["suspicion_score"] = round(total / MAX_WEIGHT, 4)
    result["flags_triggered"] = sum(1 for _, _, lbl in CHECKS if result[lbl])
    result["flag_summary"]    = " | ".join(all_details)

    t     = THRESHOLDS.get(mode, THRESHOLDS["default"])
    score = result["suspicion_score"]
    nf    = result["flags_triggered"]
    is_flagged = int(
        score >= t["honeypot_score"] or nf >= t["honeypot_flags"] or
        score >= t["suspicious_score"] or nf >= t["suspicious_flags"]
    )
    result["flagged"] = is_flagged
    result["flagged_label"] = "YES" if is_flagged else "NO"
    result["verdict"] = "FLAGGED" if is_flagged else "CLEAN"
    return result


### 2.3 I/O helper (from `flag_profiles.py`, unchanged)

In [23]:
from pathlib import Path

def iter_candidates(path: Path):
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    yield json.loads(line)
    elif suffix == ".json":
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            yield from data
        else:
            yield data
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


### 2.4 Score the raw database, then remove flagged candidates from `candidate_nodes.json`

In [24]:
import json

# "default" catches every confirmed honeypot signature; "strict" tightens
# thresholds to target only the highest-confidence fakes.
FLAG_MODE = "default"

# --- Score the RAW candidate database with the UNMODIFIED flag_profiles.py logic ---
raw_path = Path(RAW_CANDIDATES_PATH)
flag_results = [score_candidate(c, mode=FLAG_MODE) for c in iter_candidates(raw_path)]
flagged_ids  = {r["candidate_id"] for r in flag_results if r["flagged"]}

counts = {"CLEAN": 0, "FLAGGED": 0}
for r in flag_results:
    counts[r["verdict"]] += 1

total = sum(counts.values())
print(f"Scanned raw database : {total} candidates")
print(f"  CLEAN   : {counts['CLEAN']:>6}  ({counts['CLEAN']/total*100:.1f}%)")
print(f"  FLAGGED : {counts['FLAGGED']:>6}  ({counts['FLAGGED']/total*100:.1f}%)")

# --- Remove those candidate_ids from the graph-building pool (candidate_nodes JSON) ---
with open(CANDIDATES_PATH, "r") as f:
    graph_candidates = json.load(f)

candidates_clean = [c for c in graph_candidates if c["candidate_id"] not in flagged_ids]

print(f"\nGraph pool before     : {len(graph_candidates)} candidates")
print(f"Graph pool after      : {len(candidates_clean)} candidates")

# Write the filtered pool out and point the rest of the pipeline at it
FILTERED_CANDIDATES_PATH = "candidates_filtered.json"
with open(FILTERED_CANDIDATES_PATH, "w") as f:
    json.dump(candidates_clean, f, indent=4)

CANDIDATES_PATH = FILTERED_CANDIDATES_PATH


Scanned raw database : 100 candidates
  CLEAN   :     52  (52.0%)
  FLAGGED :     48  (48.0%)

Graph pool before     : 100 candidates
Graph pool after      : 52 candidates


## 3. Graph creation (from `test2.py`)

In [25]:
import json

with open(CANDIDATES_PATH, "r") as f:
    candidates = json.load(f)

all_skills = set()

for c in candidates:
    all_skills.update(
        c["skills"].keys()
    )

all_skills = sorted(all_skills)

all_assessments = set()

for c in candidates:
    all_assessments.update(
        c["assessment_scores"].keys()
    )

all_assessments = sorted(all_assessments)

numeric_fields = [

    "profile_completeness_score",

    "profile_views_received_30d",

    "applications_submitted_30d",

    "recruiter_response_rate",

    "avg_response_time_hours",

    "connection_count",

    "endorsements_received",

    "notice_period_days",

    "salary_min",

    "salary_max",

    "github_activity_score",

    "search_appearance_30d",

    "saved_by_recruiters_30d",

    "interview_completion_rate",

    "offer_acceptance_rate"
]

mins = {}
maxs = {}

for field in numeric_fields:

    values = [
        c["redrob_vector"][field]
        for c in candidates
    ]

    mins[field] = min(values)
    maxs[field] = max(values)

def skill_similarity(c1, c2):

    s1 = set(c1["skills"].keys())
    s2 = set(c2["skills"].keys())

    union = s1 | s2

    if len(union) == 0:
        return 0

    return len(s1 & s2) / len(union)


def industry_similarity(c1, c2):

    i1 = set(c1["industry_set"])
    i2 = set(c2["industry_set"])

    union = i1 | i2

    if len(union) == 0:
        return 0

    return len(i1 & i2) / len(union)


def certification_similarity(c1, c2):

    a = set(c1["certifications"])
    b = set(c2["certifications"])

    if len(a | b) == 0:
        return 1

    return len(a & b) / len(a | b)

MAX_EXP = max(
    c["years_experience"]
    for c in candidates
)

def experience_similarity(c1, c2):

    diff = abs(
        c1["years_experience"]
        -
        c2["years_experience"]
    )

    return max(
        0,
        1 - diff/MAX_EXP
    )

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def assessment_vector(candidate):

    scores = candidate["assessment_scores"]

    return np.array([

        scores.get(skill, 0)

        for skill in all_assessments

    ])

def assessment_similarity(c1, c2):

    v1 = assessment_vector(c1)
    v2 = assessment_vector(c2)

    if np.sum(v1) == 0 or np.sum(v2) == 0:
        return 0

    return cosine_similarity(
        [v1],
        [v2]
    )[0][0]


def redrob_vector(candidate):

    r = candidate["redrob_vector"]

    vec = []

    for field in numeric_fields:

        value = r[field]

        mn = mins[field]
        mx = maxs[field]

        norm = (
            value - mn
        ) / (
            mx - mn + 1e-9
        )

        vec.append(norm)

    vec.extend([

        r["open_to_work_flag"],

        r["willing_to_relocate"],

        r["verified_email"],

        r["verified_phone"],

        r["linkedin_connected"]

    ])

    return np.array(vec)


def redrob_similarity(c1, c2):

    v1 = redrob_vector(c1)
    v2 = redrob_vector(c2)

    return cosine_similarity(
        [v1],
        [v2]
    )[0][0]


def final_similarity(c1, c2):

    return (

        0.40 * skill_similarity(c1,c2)

        +

        0.15 * assessment_similarity(c1,c2)

        +

        0.10 * certification_similarity(c1,c2)

        +

        0.10 * industry_similarity(c1,c2)

        +

        0.10 * experience_similarity(c1,c2)

        +

        0.15 * redrob_similarity(c1,c2)

    )

graph = {}

for candidate in candidates:

    sims = []

    for other in candidates:

        if (
            candidate["candidate_id"]
            ==
            other["candidate_id"]
        ):
            continue

        sim = final_similarity(
            candidate,
            other
        )

        sims.append(
            (
                other["candidate_id"],
                round(sim,4)
            )
        )

    sims.sort(
        key=lambda x:x[1],
        reverse=True
    )

    graph[
        candidate["candidate_id"]
    ] = sims



with open(GRAPH_OUTPUT_PATH, "w") as f:
    json.dump(graph, f, indent=4)

print(f"Graph saved to {GRAPH_OUTPUT_PATH} with {len(graph)} nodes.")

Graph saved to /content/graphical_representation.json with 52 nodes.


## 4. Graph traversal (from `DFS.py`)

In [26]:

import json


def load_graph(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def resolve_id(graph: dict, query) -> str:
    """Accepts full id ('CAND_0071487'), numeric string/int (25, '71487'), etc."""
    query = str(query)
    if query in graph:
        return query
    padded = f"CAND_{int(query):07d}"
    if padded in graph:
        return padded
    matches = [k for k in graph if query in k]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Ambiguous id '{query}', candidates: {matches}")
    raise KeyError(f"No candidate matching '{query}' found in graph")


def dfs_chain(graph: dict, source: str, num_nodes: int = 10):
    """
    Walks a chain of `num_nodes` total nodes (including the source),
    always jumping to the current node's single most similar unvisited
    neighbor.

    Returns a list of (node, edge_weight_from_previous, hop_index).
    edge_weight is None for the starting node (hop 0).
    """
    if source not in graph:
        raise KeyError(f"'{source}' not found in graph")

    visited = {source}
    chain = [(source, None, 0)]
    current = source

    while len(chain) < num_nodes:
        neighbors = sorted(graph.get(current, []), key=lambda x: x[1], reverse=True)

        # pick the highest-weight neighbor that hasn't been visited yet
        next_node, next_weight = None, None
        for cand, weight in neighbors:
            if cand not in visited:
                next_node, next_weight = cand, weight
                break

        if next_node is None:
            # dead end: every neighbor of current node is already visited
            print(f"[stopped early] '{current}' has no unvisited neighbors left "
                  f"(reached {len(chain)}/{num_nodes} nodes).")
            break

        visited.add(next_node)
        chain.append((next_node, next_weight, len(chain)))
        current = next_node

    return chain


def print_chain(chain):
    print(f"\nGreedy DFS chain ({len(chain)} nodes):\n")
    print(f"{'Step':<6}{'Candidate':<16}{'Edge Weight':<12}")
    print("-" * 40)
    for node, weight, step in chain:
        w = "-" if weight is None else weight
        print(f"{step:<6}{node:<16}{w}")

    print("\nPath:")
    print(" -> ".join(n for n, _, _ in chain))


## 5. Run the traversal and get the list of candidate IDs

In [27]:
graph = load_graph(GRAPH_OUTPUT_PATH)
source = resolve_id(graph, SOURCE_CANDIDATE)
chain = dfs_chain(graph, source, num_nodes=NUM_NODES)
print(graph)

candidate_id_list = [node for node, _, _ in chain]
print(candidate_id_list)


{'CAND_0071487': [['CAND_0011430', 0.4098], ['CAND_0050774', 0.3772], ['CAND_0005022', 0.3685], ['CAND_0046436', 0.3656], ['CAND_0089721', 0.3642], ['CAND_0057610', 0.363], ['CAND_0079608', 0.3533], ['CAND_0038726', 0.3528], ['CAND_0098779', 0.3448], ['CAND_0096620', 0.3381], ['CAND_0061300', 0.338], ['CAND_0097670', 0.3341], ['CAND_0095052', 0.3302], ['CAND_0078252', 0.3291], ['CAND_0038563', 0.3278], ['CAND_0084579', 0.3267], ['CAND_0087238', 0.3263], ['CAND_0005190', 0.3254], ['CAND_0083283', 0.3221], ['CAND_0054898', 0.3216], ['CAND_0039962', 0.3173], ['CAND_0014793', 0.3132], ['CAND_0025516', 0.3082], ['CAND_0090714', 0.3059], ['CAND_0008294', 0.3032], ['CAND_0025670', 0.3006], ['CAND_0023472', 0.2976], ['CAND_0001907', 0.2891], ['CAND_0075417', 0.2858], ['CAND_0098995', 0.2846], ['CAND_0076016', 0.2844], ['CAND_0027798', 0.2819], ['CAND_0041664', 0.2798], ['CAND_0059820', 0.2784], ['CAND_0014146', 0.2744], ['CAND_0023594', 0.2624], ['CAND_0039495', 0.2611], ['CAND_0049080', 0.261

## 6. Candidate Reason Generation (LLM-based)

This section is a separate step: for each candidate, a local language model (Qwen2.5-0.5B-Instruct) generates a short, fact-based paragraph explaining why they're a strong match.

### 6.1 Load the DFS chain candidates (from Section 2 + Section 5)

In [28]:
import json

# Reuse the same filtered (honeypot-removed) pool from Section 2, then narrow
# it down to just the candidates that came out of the DFS chain (Section 5),
# instead of generating reasons for the entire filtered pool.
with open(FILTERED_CANDIDATES_PATH, "r") as f:
    all_filtered_candidates = json.load(f)

candidates = [c for c in all_filtered_candidates if c["candidate_id"] in candidate_id_list]

print(f"Filtered pool         : {len(all_filtered_candidates)} candidates")
print(f"DFS chain candidates  : {len(candidate_id_list)}")
print(f"Matched for reasoning : {len(candidates)}")

missing = set(candidate_id_list) - {c["candidate_id"] for c in candidates}
if missing:
    print(f"WARNING: {len(missing)} chain id(s) not found in the filtered pool: {missing}")


Filtered pool         : 52 candidates
DFS chain candidates  : 10
Matched for reasoning : 10


### 6.2 Load the language model

In [29]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

Using: cuda


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

### 6.3 Define the reason-generation function

In [30]:
def generate_reason(candidate):

    prompt = f"""
You are an AI recruitment assistant.

Candidate Profile:
{json.dumps(candidate, indent=2)}

Task:
Generate EXACTLY ONE sentence explaining why the candidate is a good match.

Rules:
- Maximum 25 words.
- Use ONLY facts from the profile.
- Include 2-4 concrete pieces of evidence.
- Evidence can include:
  • Years of experience
  • Top technical skills
  • Industry/domain
  • Assessment scores (with values)
  • Recruiter metrics (with values)
  • Certifications
- Never invent or infer anything.
- Never mention missing fields.
- Never use generic praise such as:
  "strong candidate"
  "highly qualified"
  "valuable asset"
  "excellent fit"
  "good match"
- Vary the sentence structure across candidates.
- Do NOT always start with years of experience.
- Return ONLY the sentence.

Examples:
Python, SQL, and AWS backed by 5 years of experience and Coding Assessment 92%.
Healthcare domain experience with TensorFlow, PyTorch, and Recruiter Score 88.
AWS Certified Solutions Architect with Kubernetes, Docker, and 7 years of cloud experience.
Coding Assessment 95%, Java, Spring Boot, PostgreSQL, and 4 years of FinTech experience.
React, TypeScript, Node.js, and 6 years of E-commerce experience with Recruiter Score 91.
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You generate concise evidence-based hiring reasons using only "
                "the structured candidate profile. "
                "Never hallucinate, infer, or use generic praise. "
                "Vary sentence structure naturally."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=True,
            temperature=0.35,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
        )

    answer = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return answer.strip()

### 6.4 Generate reasons for all candidates

In [31]:
results = []

for i, candidate in enumerate(candidates):

    print(f"{i+1}/{len(candidates)}")

    reason = generate_reason(candidate)

    results.append({
        "candidate_id": candidate["candidate_id"],
        "reason": reason
    })

1/10
2/10
3/10
4/10
5/10
6/10
7/10
8/10
9/10
10/10


### 6.5 Save results

In [32]:
with open("candidate_reasons.json", "w") as f:
    json.dump(results, f, indent=4)

print("Saved!")

Saved!
